In [54]:
import pandas as pd
from scipy.sparse import hstack
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

In [30]:
df = pd.read_csv("../data/cleaned/news_1.csv")

## Data preprocessing

### target class balance

In [31]:
(df["target"].value_counts()/len(df["target"])) * 100        # the classes are balanced

target
0    52.298543
1    47.701457
Name: count, dtype: float64

In [32]:
df.isnull().sum()

title      0
text       0
subject    0
target     0
dtype: int64

### duplicates dropped

In [33]:
df = df.drop_duplicates()

### strip, lowercase, and excess spaces

In [34]:
import re
df[["title", "text", "subject"]] = df[["title", "text", "subject"]].apply(lambda col: col.str.strip().str.lower().str.replace(r"\s+", " ", regex=True))

In [35]:
df["subject"].unique()

array(['politics', 'worldnews', 'politicsnews', 'news', 'government news',
       'left-news', 'us_news', 'middle-east'], dtype=object)

In [36]:
df.loc[df["subject"] == "news", "subject"] = "unknown"

### Analysing Text columns

In [37]:
samples = df["text"].sample(5)
for sample in samples:
    print(sample, "\n")

new york/singapore (reuters) - united airlines said it had resumed flights from newark, new jersey to new delhi, india on sunday, after suspending the service temporarily over concerns about poor air quality in the indian capital. ua flight 82 had been canceled on friday and saturday, data from flight tracking website flightradar24 showed, while the airline s website said it had waiver policies in place for passengers traveling to, from or through delhi until monday. ua flight 82 has resumed operations, but we will continue to monitor conditions over the next few days , a spokesman said. the third-largest u.s carrier is monitoring advisories as the new delhi region remains under a public health emergency, and is coordinating with respective government agencies, the united airlines spokesman had earlier told reuters. last week, new delhi declared a pollution emergency as toxic smog hung over the city for days, with tourism operators reporting cancellation of bookings for the christmas h

## NOTE: I am skipping other text preprocessing like removing html tags, puncations, numbers, urls, stopwords, stemming/lemmatization I want to check iF they effect or not

In [38]:
X = df.drop("target", axis=1)
y = df["target"]

In [47]:
ohe = OneHotEncoder(sparse_output=False)

subject_matrix = ohe.fit_transform(X[["subject"]])
subject_matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [48]:
cv = CountVectorizer()

title_matrix = cv.fit_transform(X["title"])
title_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 544362 stored elements and shape (44685, 20896)>

In [ ]:
cv1 = CountVectorizer()

text_matrix = cv1.fit_transform(X["text"])
text_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 9346370 stored elements and shape (44685, 122002)>

In [ ]:
# combine sparse matrices column-wise
X = hstack([title_matrix, text_matrix, subject_matrix])

<COOrdinate sparse matrix of dtype 'float64'
	with 9935417 stored elements and shape (44685, 142906)>

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y ,random_state=42)

In [58]:
NB_tfidf = MultinomialNB()

NB_tfidf.fit(X_train, y_train)
y_pred = NB_tfidf.predict(X_test)

In [59]:
accuracy_score(y_test, y_pred)

0.9712431464697325

In [60]:
confusion_matrix(y_test, y_pred)

array([[4564,  131],
       [ 126, 4116]])